# 集合パッキング問題 (Set Packing Problem)

tutorial007_exact_cover_en.ipynb と似た問題です。

集合パッキングは、計算複雑性理論と組合せ論における古典的なNP完全問題であり、Karpの21のNP完全問題の1つです。

有限集合 $S$ とその部分集合のリストがあるとします。このとき集合パッキング問題は、リストの中のいくつかの部分集合が互いに素であるか(言い換えれば、どの2つも要素を共有しないか)を問う問題です。

https://en.wikipedia.org/wiki/Set_packing

## インストール

まずblueqatをインストールしてください。

```bash
pip install git+https://github.com/blueqat/blueqatSDK
```

ライブラリをインポートし、問題のインスタンスを準備します。

In [1]:
import numpy as np
from blueqat.utils import Vqe, QaoaAnsatz
from blueqat.utils import qubo_bit as q

## QUBOの作成

今回のコスト関数は

$ E = E_{A} + E_{B} $

であり、$E_{A}, E_{B}$ はそれぞれ次のように定義されます。

$ E _ { A } = A \sum _ { i , j : V _ { i } \cap V _ { j } \neq \emptyset } x _ { i } x _ { j }$

$E _ { B } = - B \sum _ { i } x _ { i }$

$A, B$ については $A > B$ である必要があります。

In [2]:
A = 1.0
B = 0.9

def get_qubo(V):
    Q = 0

    for i in range(len(V)):
        for j in range(i, len(V)):
            if i == j:
                Q += -B*q(i)*q(i)
            elif len(V[i]) + len(V[j]) != len( set(V[i]) | set(V[j]) ):
                Q +=  A*q(i)*q(j)

    return Q

結果を表示するための関数です。

In [3]:
def show_answer(list_x, energies = None, show_graph = False):
    print("Result x:", list_x)
    text = ""
    for i in range(len(list_x)):
        if(list_x[i]):
            text += str(V[i])
    print("Picked {} group(s): {}".format(sum(list_x), text))
    if energies is not None:
        print("Energy:", a.E[-1])
    if show_graph:
        plt.plot(a.E)
        plt.show()

実際に実行してみましょう。

In [4]:
V = [ [1,2], [3,4,5,6], [7,8,9,10], [1,3,5], [10], [7,9], [2,4,6,8], [1,2,3,4,5,6,8,10] ]

h = get_qubo(V)
# Note: in the new SDK, `step` is the number of variational QAOA layers
# actually optimized by gradient descent (unlike the old Trotter-schedule
# semantics), so a small step count is used here to keep runtime reasonable.
step = 5
result = Vqe(QaoaAnsatz(h, step)).run(max_iter=100)
print(result.most_common(12))

/Users/yuichirominato/blueqatSDK/.claude/worktrees/determined-mahavira-bf713e/blueqat/utils.py:399: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:823.)
  total_matrix = torch.sparse_coo_tensor(torch.empty((2, 0), dtype=torch.int64, device=device), torch.empty(0, dtype=torch.complex128, device=device), (dim, dim))


(((1, 1, 0, 0, 1, 1, 0, 0), 0.07014558499616247), ((0, 0, 0, 1, 1, 1, 1, 0), 0.06864794433099701), ((1, 1, 1, 0, 1, 1, 0, 0), 0.05391591751132892), ((0, 0, 0, 1, 1, 1, 0, 0), 0.04096119638147674), ((1, 1, 0, 0, 1, 1, 1, 0), 0.02865943668379661), ((1, 1, 0, 0, 0, 1, 0, 0), 0.023992191711143317), ((0, 0, 0, 0, 1, 1, 0, 1), 0.020758682161461598), ((0, 0, 0, 0, 0, 1, 0, 1), 0.019586343052938188), ((1, 1, 1, 0, 1, 0, 0, 0), 0.018141203711145543), ((0, 0, 0, 1, 0, 1, 0, 1), 0.017607974819971995), ((1, 0, 1, 1, 1, 1, 0, 0), 0.017370615050652266), ((0, 1, 1, 1, 1, 1, 0, 0), 0.01737061505065224))


正解は {1,3,5},{10},{7,9},{2,4,6,8} です。実行するたびに異なる答えが選ばれることもあります。

この集合パッキング問題は、最大独立集合(MIS)問題と類似しています。この2つの問題は本質的に同じものです。